# What can edgartools actually do?

This notebook is a hands-on tour of **edgartools** — the library we use to pull
SEC filings — run against one real filing: **American Express's fiscal-2022
10-K**. The goal is for you to *see* what it gives us, especially the difference
between **flat text** (what our RAG pipeline reads today) and **structured data**
(numbers in clean tables, which is the fix for our table-retrieval misses).

We'll go area by area:

1. **Fetch & metadata** — find the right 10-K, read its basics
2. **Narrative sections** — the prose (Business, Risk Factors, MD&A) we chunk today
3. **Structured financials** — the income statement / balance sheet as real tables
4. **XBRL facts** — every reported number, tagged and machine-readable
5. ⭐ **The geography table** — the exact table our agent fails on, pulled as clean data
6. **RAG extras** — edgartools' own table-aware chunker & search

> Run top to bottom. The first cell downloads the filing (a few seconds; it's
> cached after that), and everything else reuses it.

## Setup — load the filing once

One trick to know up front: **a 10-K filed in early 2023 covers fiscal year
2022.** So to get "the fiscal 2022 report," you filter by `period_of_report`
(what the report *covers*), not `filing_date` (when it was *submitted*). Watch the
two 10-Ks that come back below — same company, different fiscal years.

In [ ]:
from sec_filings.config import settings
from edgar import Company, set_identity

set_identity(settings.edgar_identity)   # SEC asks you to identify your traffic

# Two 10-Ks come back: the one filed in 2023 covers FY2022; the one filed in 2022 covers FY2021.
filings = Company("AXP").get_filings(form="10-K", year=[2022, 2023])
for x in filings:
    print(f"{x.form}  acc={x.accession_no}  filed={x.filing_date}  period={x.period_of_report}")

# Pick by what it COVERS, not when it was filed:
filing = next(x for x in filings if str(x.period_of_report).startswith("2022"))
tenk = filing.obj()   # parse the raw filing into a rich TenK object — the gateway to everything below
print("\nPicked:", filing.accession_no, "->", type(tenk).__name__)

## 1. Metadata & raw renderings

Basic facts about the filing, plus the three ways edgartools can hand you the raw
document: `text()`, `markdown()`, and `html()` (all plain Python strings). Note
`markdown()` *keeps the heading structure* — useful if you ever want to chunk by
section without writing your own HTML parser.

In [ ]:
print("company:", filing.company, "| cik:", filing.cik)
print("accession:", filing.accession_no, "| form:", filing.form)
print("filing_date:", filing.filing_date, "| period_of_report:", filing.period_of_report)
print("primary_document:", filing.primary_document)
print("url:", filing.url)
print()

txt = filing.text()        # plain text
mkd = filing.markdown()    # markdown, headings preserved
html = filing.html()       # full original HTML
print("text len:", len(txt), "| markdown len:", len(mkd), "| html len:", len(html))
print("markdown keeps headings ->", [ln for ln in mkd.splitlines() if ln.strip()][:2])

## 2. Narrative sections — the prose we chunk today

This is the layer our current pipeline consumes. edgartools already split the
10-K into named sections for us — no HTML scraping, no "ITEM 1." regex. The big
narrative sections come back as plain strings, you can grab any Item by label,
and it'll print the document outline it inferred.

In [ ]:
# Named prose sections are plain strings — ready to chunk / embed.
print(tenk.business[:300])
print()
print("business chars     :", len(tenk.business))
print("risk_factors chars :", len(tenk.risk_factors))
print("MD&A chars         :", len(tenk.management_discussion))

In [ ]:
# Which Items does this filing actually contain? (sorted+unique — the raw list has dupes)
print("Items present:", sorted(set(tenk.items)))
print()
# Grab any Item by its label:
print(tenk["Item 1A"][:220])

In [ ]:
# The document outline edgartools inferred (Parts -> Items):
import io
from rich.console import Console

buf = io.StringIO()
Console(file=buf, width=78, no_color=True).print(tenk.get_structure())
print(buf.getvalue())

In [ ]:
# Keyword search within the filing's prose:
res = tenk.grep("Membership Rewards")
print("matches:", len(res.matches))
print(str(res.matches[0])[:160])

## 3. Structured financials — numbers in real tables

Here's where it gets good. Instead of a wall of text, edgartools gives you each
financial statement as a **structured table** (a pandas DataFrame): one row per
line item, with a clean `label`, the value per fiscal year, and metadata. A
question like *"what was the net profit margin?"* becomes a **cell lookup**, not a
guess over flattened text.

In [ ]:
fin = tenk.financials
inc = fin.income_statement().to_dataframe()   # income_statement() is a METHOD; .to_dataframe() -> pandas
print(type(inc).__name__, inc.shape)
# 'label' = the row name, '2022-12-31 (FY)' = that year's number
print(inc[["label", "2022-12-31 (FY)"]].dropna().head(6).to_string(index=False))

In [ ]:
# Pull specific line items by name and compute a margin — exact, not hallucinated.
def get_line(df, text, col="2022-12-31 (FY)"):
    # dimension==False keeps the top-line total, not a segment break-out of it
    rows = df[(df["label"].str.contains(text, case=False, na=False)) & (df["dimension"] == False)]
    rows = rows[rows[col].notna()]
    return None if rows.empty else float(rows.iloc[0][col])

total_rev = get_line(inc, "^Total revenues net of interest expense$")
net_inc = get_line(inc, "^Net income$")
print(f"Total revenues net of interest expense: {total_rev:,.0f}")
print(f"Net income:                             {net_inc:,.0f}")
print(f"Net profit margin: {net_inc / total_rev:.1%}")

In [ ]:
# Convenience helpers return common totals as plain floats (no DataFrame wrangling):
print("net_income       :", fin.get_net_income())
print("total_assets     :", fin.get_total_assets())
print("total_liabilities:", fin.get_total_liabilities())
print("equity           :", fin.get_stockholders_equity())
print("op_cash_flow     :", fin.get_operating_cash_flow())
# GOTCHA: for AXP (a bank) get_revenue() returns the 'Discount revenue' sub-line (30.7B),
# NOT the 52.86B headline total. For banks, pull the explicit total label from the DataFrame.
print("get_revenue (careful — sub-line, not the headline):", fin.get_revenue())

## 4. XBRL facts — every number, tagged and machine-readable

Underneath the tables is **XBRL**: every reported number is a "fact" carrying a
standard concept tag, its period, value, and unit. This is how you get an *exact*
figure *with a citation* — far beyond our current FMP tool (which only knows
revenue + R&D). You can query by the exact tag, or discover the tag from a label
when you don't know it.

In [ ]:
xb = filing.xbrl()   # the XBRL layer of THIS filing (covers 2020-2022)

# Query one concept -> value per period, with unit. is_dimensioned=False is the company total.
df = (xb.query()
        .by_concept("us-gaap:RevenuesNetOfInterestExpense")
        .to_dataframe("concept", "label", "value", "period_end", "unit_ref", "is_dimensioned"))
total = df[~df["is_dimensioned"]]
for _, r in total.iterrows():
    print(f"{r['period_end']}  {int(r['value']) / 1e9:6.2f}B {r['unit_ref']}  {r['label']}")

In [ ]:
# Don't know the tag? Search by the human label instead of guessing.
d = (xb.query()
       .by_label("Net income", exact=False)
       .to_dataframe("concept", "label", "value", "period_end", "is_dimensioned"))
d = d[~d["is_dimensioned"]].drop_duplicates(["concept", "period_end"])
print(d[d["period_end"] == "2022-12-31"][["concept", "label", "value"]].head(3).to_string(index=False))

## 5. ⭐ The geography table — the exact thing our agent fails on

This is the payoff. Our RAG agent can't answer *"what geographies does AmEx
operate in?"* because that table gets shredded when we flatten it to text. But the
filing **tagged** that table in XBRL: the value is `RevenuesNetOfInterestExpense`,
and each region is a "member" on the **geographic axis**. One query gives us a
clean region → revenue mapping — and the six numbers match the filing exactly.

In [ ]:
xb = filing.xbrl()

# Revenue, broken out along the geographic axis:
geo = (xb.query()
         .by_concept("us-gaap:RevenuesNetOfInterestExpense")
         .by_dimension("srt_StatementGeographicalAxis")
         .to_dataframe())
geo = geo[geo["period_end"] == "2022-12-31"]   # FY2022 only

print("Region              $millions")
print("-" * 30)
for r in geo.itertuples():
    print(f"{r.dimension_member_label:18s} {int(r.numeric_value) // 1_000_000:>8,}")

# The consolidated TOTAL is the SAME concept with NO dimension:
tot = xb.query().by_concept("us-gaap:RevenuesNetOfInterestExpense").to_dataframe()
tot = tot[(tot["period_end"] == "2022-12-31") & (~tot["is_dimensioned"])]
print(f"{'Consolidated':18s} {int(tot.iloc[0].numeric_value) // 1_000_000:>8,}")

And the table's **lead-in caption** is retrievable as plain text too — which is
exactly what you'd embed so a search for "geographic regions" finds it, while the
structured numbers above become the grounded answer. Both halves of the idea,
confirmed:

In [ ]:
hits = tenk.grep("following table presents our total revenues net of interest expense")
primary = next(m for m in hits.matches if str(m).startswith("primary:"))
caption = str(primary).split("GEOGRAPHIC OPERATIONS", 1)[-1].strip()
print(caption[:220])

## 6. RAG extras — edgartools' own table-aware chunker & search

edgartools ships pieces that overlap with our pipeline. The important one:
`chunked_document` is its **own chunker, and it's table-aware** — it keeps each
table whole as one chunk (116 of them here) instead of slicing it into half-rows
the way our sentence-based chunker does. It can render a table chunk to markdown
with the numbers intact.

In [ ]:
cd = tenk.chunked_document
# A chunk is a list of "blocks"; count chunks that contain a table block:
n_tables = sum(any("Table" in type(b).__name__ for b in ch) for ch in cd.chunks)
print(f"chunks={len(cd.chunks)}  avg_chars={cd.average_chunk_size()}  table_chunks={n_tables}")

In [ ]:
# Render a financial table chunk to markdown (alignment is rough, but numbers survive):
table_blocks = list(cd.tables())   # .tables() is a generator -> list it
fin_tbl = next(t for t in table_blocks if "$" in t.to_markdown() and "2022" in t.to_markdown())
print("\n".join(fin_tbl.to_markdown().splitlines()[:6]))

In [ ]:
# to_context(): a short LLM-ready SUMMARY header (~900 chars) — NOT full text, NOT a chunker.
print(tenk.to_context(detail="standard")[:300])

In [ ]:
# search_filings(): live full-text search ACROSS all of EDGAR (not just this filing).
from edgar import search_filings

res = search_filings('"net interest margin"', forms="10-K", limit=5)
print("total filings:", res.total)
for h in res.results[:3]:
    print(f"  {h.score:>6.2f} {h.form} {h.company[:38]} {h.filed}")

## What this means for our agent

You've now seen the whole toolbox. Two concrete upgrades fall out of it:

- **A structured-data tool.** For numbers and tables — income statement, balance
  sheet, *and the geographic segment table* — we can fetch exact, machine-readable
  values straight from `tenk.financials` / `filing.xbrl()`. No embedding, no
  shredding, and citations come for free. This is the real fix for the
  table-retrieval misses, and it lets us retire the FMP tool (which only knew two
  concepts).
- **Table-aware chunking** for the prose path. Where we *do* still chunk text,
  `chunked_document` keeps tables whole instead of slicing them — so a question
  whose answer lives in a smaller, non-XBRL table still retrieves cleanly.

Plus the **`period_of_report` rule** from the setup cell, which keeps the agent
from ever answering out of the wrong year's report.

The two paths line up with the two kinds of question: **structured fetch** for
"what was the number," **retrieval** for "tell me about the business." That's the
routing your agent already does — we're just giving it much better tools to route
*to*.